<a href="https://colab.research.google.com/github/albertoliuzzo/progetti_master_AI_Engineering/blob/main/Deep_learning_riconoscimento_di_animali_per_auto_a_guida_autonoma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pickle
import numpy as np
import os

def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

# percorso alla cartella
data_dir = "cifar-10-batches-py"

# classi
label_names = unpickle(os.path.join(data_dir, 'batches.meta'))[b'label_names']
label_names = [label.decode('utf-8') for label in label_names]
print("Classi:", label_names)

# carico i batch di training
x_train = []
y_train = []

for i in range(1, 6):
    batch = unpickle(os.path.join(data_dir, f"data_batch_{i}"))
    x_train.append(batch[b'data'])
    y_train += batch[b'labels']

x_train = np.concatenate(x_train).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
y_train = np.array(y_train)

# carico i batch di test
test_batch = unpickle(os.path.join(data_dir, "test_batch"))
x_test = test_batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
y_test = np.array(test_batch[b'labels'])

# normalizzazione
x_train = x_train / 255.0
x_test = x_test / 255.0

#stampo un'immagine per prova
i = 1
plt.imshow(x_train[i])
plt.title(f"Classe: {label_names[y_train[i]]}")
plt.show()

# Definisco le classi rilevanti (veicoli,animali)
# classi da mantenere
animal_classes = ["bird", "cat", "deer", "dog", "frog", "horse"]
vehicle_classes = ["airplane", "automobile", "ship", "truck"]

# mappatura da nome a etichetta originale
label_to_index = {label: i for i, label in enumerate(label_names)}

# estraggo gli indici numerici originali delle due categorie
animal_indices = [label_to_index[label] for label in animal_classes]
vehicle_indices = [label_to_index[label] for label in vehicle_classes]

# Filtro per avere solo 2 etichette e rimappo le immagini in vettori (n, 32, 32, 3) sia per test che per training set
# nuove liste filtrate
x_train_filtered = []
y_train_filtered = []
x_test_filtered = []
y_test_filtered = []

for i in range(len(x_train)):
    label = y_train[i]
    if label in animal_indices:
        x_train_filtered.append(x_train[i])
        y_train_filtered.append(0)  # animale
    elif label in vehicle_indices:
        x_train_filtered.append(x_train[i])
        y_train_filtered.append(1)  # veicolo

for i in range(len(x_test)):
    label = y_test[i]
    if label in animal_indices:
        x_test_filtered.append(x_test[i])
        y_test_filtered.append(0)
    elif label in vehicle_indices:
        x_test_filtered.append(x_test[i])
        y_test_filtered.append(1)

# converto in array numpy
x_train_filtered = np.array(x_train_filtered)
y_train_filtered = np.array(y_train_filtered)
x_test_filtered = np.array(x_test_filtered)
y_test_filtered = np.array(y_test_filtered)

# costruisco la CNN con 3 layer convoluzionali (+pooling) e 1 layer denso
model = Sequential([
    # primo blocco convoluzionale
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    # secondo blocco
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # terzo blocco
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # layer fully-connected
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # per classificazione binaria veicolo-animale
])

# compilo il modello
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy', 'Precision'])

# addestro il modello
history = model.fit(x_train_filtered, y_train_filtered, epochs=10, batch_size=64)

# valuto accuratezza e precisione
loss, accuracy, precision = model.evaluate(x_test_filtered, y_test_filtered)
print(f"Accuratezza: {accuracy:.2f}, Precisione: {precision:.2f}")

# Costruisco la confusion matrix
# predizioni
y_pred_probs = model.predict(x_test_filtered)

# converto in etichette binarie (0 o 1)
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

cm = confusion_matrix(y_test_filtered, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Animale", "Veicolo"])

disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix su test set")
plt.show()

# Identificazione di pattern di errore
original_labels = np.array([y for i, y in enumerate(y_test) if y_test[i] in animal_indices + vehicle_indices]) #etichette originali prima della riclassificazione
error_classes = original_labels[wrong_indices]
error_class_names = [label_names[i] for i in error_classes]

from collections import Counter
print("Errori per classe originale:", Counter(error_class_names))

# Esame delle immagini errate e proposta di miglioramenti
# trovo gli indici delle predizioni errate
wrong_indices = np.where(y_pred != y_test_filtered)[0]
print(f"Numero di errori: {len(wrong_indices)}")

# mostro le prime 10 immagini sbagliate
num_to_show = 10
plt.figure(figsize=(15, 6))

for i, idx in enumerate(wrong_indices[:num_to_show]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test_filtered[idx])
    true_label = "Animale" if y_test_filtered[idx] == 0 else "Veicolo"
    pred_label = "Animale" if y_pred[idx] == 0 else "Veicolo"
    plt.title(f"Vero: {true_label}\nPredetto: {pred_label}")
    plt.axis('off')

plt.suptitle("Immagini sbagliate: etichette vere vs predette", fontsize=14)
plt.tight_layout()
plt.show()

#L’analisi della confusion matrix e delle predizioni errate mostra che il modello tende a confondere alcune immagini di animali con veicoli specialmente quando il soggetto è piccolo (bird) e immagini di aerei con animali
#Per migliorare le prestazioni si possono utilizzare tecniche di data augmentation, modelli pre-addestrati e una maggiore varietà di immagini di training